In [5]:
import pandas as pd
import numpy as np

# Create Employee dataframe
employee = pd.DataFrame({
    "ID": ["A001","A002","A003","A004","A005"],
    "Name": ["John Alter","Alice Luxumberg","Tom Sabestine","Nina Adgra","Amy Johny"],
    "Gender": ["M","F","M","F","F"],
    "City": ["Paris","London","Berlin","Newyork","Madrid"],
    "Age": [25,27,29,31,30]
})

# Create Seniority dataframe
seniority = pd.DataFrame({
    "ID": ["A001","A002","A003","A004","A005"],
    "Designation Level": [2,2,3,2,3]
})

# Create Project dataframe
project = pd.DataFrame({
    "ID": ["A001","A002","A003","A004","A005","A002","A005","A003","A001","A003","A001","A004","A004","A005"],
    "Project": ["Project1","Project2","Project3","Project4","Project5","Project6","Project7","Project8","Project9","Project10","Project11","Project12","Project13","Project14"],
    "Cost": [1002000,2000000,4500000,5500000,np.nan,680000,400000,350000,np.nan,300000,2000000,1000000,3000000,200000],
    "Status": ["Finished","Ongoing","Finished","Ongoing","Finished","Failed","Finished","Failed","Ongoing","Finished","Failed","Ongoing","Finished","Finished"]
})

#Create CSV files
employee.to_csv("Employee_table.csv", index=False)
seniority.to_csv("Seniority_table.csv", index=False)
project.to_csv("Project_table.csv", index=False)

In [6]:
# Read the project dataset
proj_df = pd.read_csv("Project_table.csv")

# Initialize variables
total_cost = 0
value_counter = 0

# Loop through dataframe rows
for index in range(len(proj_df)):
    
    # Check if cost value is available
    if pd.notna(proj_df.at[index, 'Cost']):
        total_cost = total_cost + proj_df.at[index, 'Cost']
        value_counter = value_counter + 1
    
    # If cost is missing, replace with running average
    else:
        proj_df.at[index, 'Cost'] = total_cost / value_counter

# Display updated dataframe
print(proj_df)

      ID    Project          Cost    Status
0   A001   Project1  1.002000e+06  Finished
1   A002   Project2  2.000000e+06   Ongoing
2   A003   Project3  4.500000e+06  Finished
3   A004   Project4  5.500000e+06   Ongoing
4   A005   Project5  3.250500e+06  Finished
5   A002   Project6  6.800000e+05    Failed
6   A005   Project7  4.000000e+05  Finished
7   A003   Project8  3.500000e+05    Failed
8   A001   Project9  2.061714e+06   Ongoing
9   A003  Project10  3.000000e+05  Finished
10  A001  Project11  2.000000e+06    Failed
11  A004  Project12  1.000000e+06   Ongoing
12  A004  Project13  3.000000e+06  Finished
13  A005  Project14  2.000000e+05  Finished


In [9]:
employee = pd.read_csv("Employee_table.csv")

employee[['First Name','Last Name']] = employee['Name'].str.split(" ", expand=True)

employee.drop('Name', axis=1, inplace=True)

print(employee)

     ID Gender     City  Age First Name  Last Name
0  A001      M    Paris   25       John      Alter
1  A002      F   London   27      Alice  Luxumberg
2  A003      M   Berlin   29        Tom  Sabestine
3  A004      F  Newyork   31       Nina      Adgra
4  A005      F   Madrid   30        Amy      Johny


In [11]:
seniority = pd.read_csv("Seniority_table.csv")

Final = pd.merge(employee, seniority, on="ID")

Final = pd.merge(Final, project, on="ID")

In [12]:
Final['Bonus'] = np.where(Final['Status']=="Finished", Final['Cost']*0.05, 0)

In [13]:
Final.loc[Final['Status']=="Failed", 'Designation Level'] -= 1

Final = Final[Final['Designation Level'] <= 4]

In [14]:
Final['First Name'] = np.where(Final['Gender']=="M",
                              "Mr. " + Final['First Name'],
                              "Mrs. " + Final['First Name'])

Final.drop('Gender', axis=1, inplace=True)

In [15]:
Final.loc[Final['Age']>29, 'Designation Level'] += 1

In [16]:
TotalProjCost = Final.groupby(['ID','First Name'])['Cost'].sum().reset_index()

TotalProjCost.rename(columns={'Cost':'Total Cost'}, inplace=True)

print(TotalProjCost)

     ID  First Name  Total Cost
0  A001    Mr. John   3002000.0
1  A002  Mrs. Alice   2680000.0
2  A003     Mr. Tom   5150000.0
3  A004   Mrs. Nina   9500000.0
4  A005    Mrs. Amy    600000.0


In [17]:
result = Final[Final['City'].str.contains("o", case=False)]

print(result)

      ID     City  Age  First Name  Last Name  Designation Level    Project  \
3   A002   London   27  Mrs. Alice  Luxumberg                  2   Project2   
4   A002   London   27  Mrs. Alice  Luxumberg                  1   Project6   
8   A004  Newyork   31   Mrs. Nina      Adgra                  3   Project4   
9   A004  Newyork   31   Mrs. Nina      Adgra                  3  Project12   
10  A004  Newyork   31   Mrs. Nina      Adgra                  3  Project13   

         Cost    Status     Bonus  
3   2000000.0   Ongoing       0.0  
4    680000.0    Failed       0.0  
8   5500000.0   Ongoing       0.0  
9   1000000.0   Ongoing       0.0  
10  3000000.0  Finished  150000.0  
